# Evaluation notebook

This notebook is used to evaluate the MLP and RF models that have been trained. The models are loaded along with all of the ice data in Peru, the probability threshold is set to ensure the predicted melt area matches the actual melt area observed, and the overlap percentage of the prediction and the actual melt is calculated.

In [1]:
import joblib
import numpy as np
import pandas as pd
import gc
import time
from pathlib import Path
from sklearn.metrics import confusion_matrix

In [2]:
LOCAL_DIR = Path("C:\\Users\\admin\\Documents\\Glacier Project")
REGIONS = ['R1a', 'R1b', 'R2', 'R3']
RANDOM_STATE = 42

# LOAD DATA

dfs = []
for r in REGIONS:
    df = pd.read_parquet(LOCAL_DIR / f'Merged\\{r.lower()}_combined.parquet')
    df['region'] = r
    dfs.append(df)
df_all = pd.concat(dfs, ignore_index=True)
ae_cols = [c for c in df_all.columns 
           if c.startswith('A') and len(c) == 3 and c[1:].isdigit()]

In [3]:
# CALCULATING OVERLAP PERCENTAGE - RF AND MLP

MODEL_TYPE = 'RF'  # 'RF' or 'MLP'
gc.collect()


MODEL_NAME = 'RF_EASD_NOSTRAT'
# Ensure scaler is included if testing an MLP model.
SCALER_NAME = ''
# Ensure feature columns match those used in training.
FEATURE_COLS = ['elevation', 'edge_distance', 'aspect', 'slope']

if MODEL_TYPE == 'MLP':
    model = joblib.load(LOCAL_DIR / f'Saved_Models\\MLPs\\{MODEL_NAME}.joblib')
    scaler = joblib.load(LOCAL_DIR / f'Saved_Models\\MLPs\\{SCALER_NAME}.joblib')

    # Spatial overlap on Peru-wide
    print('\nPredicting on full Peru-wide population...')
    probs = np.zeros(len(df_all), dtype=np.float32)
    CHUNK = 500_000
    for i in range(0, len(df_all), CHUNK):
        end = min(i + CHUNK, len(df_all))
        X_chunk = df_all[FEATURE_COLS].iloc[i:end].values
        X_chunk_scaled = scaler.transform(X_chunk)
        probs[i:end] = model.predict_proba(X_chunk_scaled)[:, 1]
        del X_chunk, X_chunk_scaled

elif MODEL_TYPE == 'RF':
    model = joblib.load(LOCAL_DIR / f'Saved_Models\\RFs\\{MODEL_NAME}.joblib')

    # Spatial overlap on Peru-wide
    print('\nPredicting on full Peru-wide population...')
    probs = np.zeros(len(df_all), dtype=np.float32)
    CHUNK = 500_000
    for i in range(0, len(df_all), CHUNK):
        end = min(i + CHUNK, len(df_all))
        X_chunk = df_all[FEATURE_COLS].iloc[i:end].values
        probs[i:end] = model.predict_proba(X_chunk)[:, 1]
        del X_chunk

n_actual_melt = (df_all['melt_label'] == 1).sum()
top_indices = np.argsort(probs)[::-1][:n_actual_melt]
predicted_melt = np.zeros(len(probs), dtype=bool)
predicted_melt[top_indices] = True
actual_melt = df_all['melt_label'].values == 1
n_intersect = (predicted_melt & actual_melt).sum()
overlap_pct = n_intersect / n_actual_melt * 100
iou = n_intersect / (predicted_melt | actual_melt).sum()

print(f'\n=== {MODEL_TYPE} OVERLAP ANALYSIS ===')
print(f'  Overlap: {overlap_pct:.2f}%')
print(f'  IoU:     {iou:.4f}')


Predicting on full Peru-wide population...

=== RF OVERLAP ANALYSIS ===
  Overlap: 73.20%
  IoU:     0.5773


In [4]:
# GENERATE PREDICTIONS AND SAVE PARQUET

CHUNK_SIZE = 500_000


# ============================================================
# IDENTIFY FEATURE SETS
# ============================================================
ae_cols = [c for c in df_all.columns 
           if c.startswith('A') and len(c) == 3 and c[1:].isdigit()]

# Adjust these to match whichever models you want predictions from.
# Each entry: (output_column_name, model_path, scaler_path, feature_cols)
MODELS_TO_RUN = [
    (
        'prob_mlp_ae64_nostrat',
        LOCAL_DIR / 'Saved_Models\\MLPs\\MLP_AE64_NOSTRAT.joblib',
        LOCAL_DIR / 'Saved_Models\\MLPs\\MLP_AE64_NOSTRAT_SCALER.joblib',
        ae_cols                          # AE only
    ),
    (
        'prob_rf_easd_nostrat',
        LOCAL_DIR / 'Saved_Models\\RFs\\RF_EASD_NOSTRAT.joblib',
        '',
        ['elevation', 'edge_distance', 'aspect', 'slope']
    ),
]

# ============================================================
# PREDICT FOR EACH MODEL IN CHUNKS
# ============================================================
for col_name, model_path, scaler_path, feature_cols in MODELS_TO_RUN:
    print(f'\nPredicting: {col_name}')
    
    if not model_path.exists():
        print(f'  SKIPPING — model file not found: {model_path}')
        continue
    
    model = joblib.load(model_path)
    scaler = joblib.load(scaler_path) if scaler_path else None
    
    # For RF, load feature cols from saved txt if not provided
    if feature_cols is None:
        txt_path = LOCAL_DIR / 'mlp_feature_cols.txt'
        if txt_path.exists():
            with open(txt_path) as f:
                feature_cols = [line.strip() for line in f if line.strip()]
        else:
            print(f'  SKIPPING — no feature list found')
            continue
    
    # Verify all columns exist
    missing = [c for c in feature_cols if c not in df_all.columns]
    if missing:
        print(f'  WARNING: missing columns: {missing[:5]}')
        continue
    
    probs = np.zeros(len(df_all), dtype=np.float32)
    for i in range(0, len(df_all), CHUNK_SIZE):
        end = min(i + CHUNK_SIZE, len(df_all))
        X_chunk = df_all[feature_cols].iloc[i:end].values
        if scaler is not None:
            X_chunk = scaler.transform(X_chunk)
        probs[i:end] = model.predict_proba(X_chunk)[:, 1]
        del X_chunk
    
    df_all[col_name] = probs
    del model, scaler
    gc.collect()
    print(f'  Done. Prob mean: {probs.mean():.3f}')




Predicting: prob_mlp_ae64_nostrat
  Done. Prob mean: 0.274

Predicting: prob_rf_easd_nostrat
  Done. Prob mean: 0.317


In [5]:
# ============================================================
# SAVE PREDICTIONS PARQUET
# ============================================================
output_cols = ['region', 'lon', 'lat', 'melt_label', 'edge_distance'] + \
              [col for col, _, _, _ in MODELS_TO_RUN 
               if col in df_all.columns]

print(f'\nSaving predictions with columns: {output_cols}')
df_all[output_cols].to_parquet(
    LOCAL_DIR / 'Predictions\\predictions_full_peru.parquet', index=False)
print(f'Saved {len(df_all):,} rows to predictions_full_peru.parquet')


Saving predictions with columns: ['region', 'lon', 'lat', 'melt_label', 'edge_distance', 'prob_mlp_ae64_nostrat', 'prob_rf_easd_nostrat']
Saved 15,242,639 rows to predictions_full_peru.parquet
